# 03 – Modelling

Choose the features and build a sklearn pipeline that prepares the data. Imputing and scaling are learned from the training data only, so the test result stays honest.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

from src.config import PROCESSED, FIGURES

## Load data and target

In [ ]:
df = pd.read_csv(PROCESSED / "clean_dataset_numeric.csv")

TARGET = "short_sleep"
print(df.shape)
df[TARGET].value_counts()

## Columns left out of the model

In [ ]:

DROP_COLS = [
    "sleep_hours_per_night",
    "past_month_felt_rested",
    "past_month_trouble_falling_asleep",
    "past_month_trouble_staying_asleep",
    "survey_weight",
    "household_id",
]

X = df.drop(columns=DROP_COLS + [TARGET])
y = df[TARGET]

## Feature types

In [ ]:

NUM_COLS = [
    "age",                        # years, 18-29
    "weekly_work_hours",          # hours per week
    "kids_household",             # number of children in the household
    "family_income",              # income-to-poverty ratio, ordered 1-14
    "food_security",              # 1 high ... 4 very low (ordered)
    "activity_guideline_status",  # 1 inactive ... 3 sufficiently active (ordered)
    "anxiety_frequency",          # ordered 1-5
    "depression_frequency",       # ordered 1-5
    "serious_distress",           # 1 / 2, two values (check the codebook for which is yes)
    "sex",                        # 1 male, 2 female (two values, works like yes/no)
    "housing_cost",               # 1 yes, 0 no
    "high_bloodpressure",         # 1 yes, 0 no
    "had_asthma",                 # 1 yes, 0 no
    "had_arthritis",              # 1 yes, 0 no
    "had_diabetes",               # 1 yes, 0 no
]

CAT_COLS = [
    "education_level",
]

missing = set(X.columns) - set(NUM_COLS) - set(CAT_COLS)
extra = (set(NUM_COLS) | set(CAT_COLS)) - set(X.columns)
assert not missing, f"Columns in X but not in any list: {missing}"
assert not extra, f"Columns in a list but not in X: {extra}"

## Preprocessing and `make_pipeline`

In [ ]:
num_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

cat_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", num_pipe, NUM_COLS),
    ("cat", cat_pipe, CAT_COLS),
])


def make_pipeline(model):
    """Preprocessing + model. Each call gets its own copy of the preprocessing."""
    return Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", model),
    ])

## Train/Test split

In [ ]:
X_train, X_test, y_train, y_test, y_train_reg, y_test_reg = train_test_split(
    X, y, y_reg, test_size=0.2, stratify=y, random_state=42)

print("Train:", X_train.shape, " Test:", X_test.shape)
print(f"Short sleep in train: {y_train.mean():.1%}")
print(f"Short sleep in test:  {y_test.mean():.1%}")

## Baseline
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.pipeline import Pipeline


base_clf = Pipeline([
    ("prepare", preprocess),
    ("model", DummyClassifier(strategy="most_frequent")),
])
base_clf.fit(X_train, y_train)
y_pred_base = base_clf.predict(X_test)


base_reg = Pipeline([
    ("prepare", preprocess),
    ("model", DummyRegressor(strategy="mean")),
])
base_reg.fit(X_train, y_train_reg)
y_pred_base_reg = base_reg.predict(X_test)

##Evaluation function for models

from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)

def evaluate_classification(name, y_true, y_pred, pos_label=1):
    row = {"model": name}
    row["accuracy"]  = accuracy_score(y_true, y_pred)
    row["precision"] = precision_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    row["recall"]    = recall_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    row["f1"]        = f1_score(y_true, y_pred, pos_label=pos_label, zero_division=0)
    return row

def evaluate_regression(name, y_true, y_pred):
    row = {"model": name}
    row["mae"]  = mean_absolute_error(y_true, y_pred)
    row["rmse"] = np.sqrt(mean_squared_error(y_true, y_pred))
    row["r2"]   = r2_score(y_true, y_pred)
    return row

In [ ]:
class_models = {
    # "Name shown in the table": the predictions from that model's .predict(X_test)
    "Baseline (most frequent)": y_pred_base
}

class_results = []
for name, preds in class_models.items():
    class_results.append(evaluate_classification(name, y_test, preds))

class_df = pd.DataFrame(class_results)
print(class_df.round(3))



reg_models = {
    # "Name shown in the table": the predictions from that model's .predict(X_test) 
    "Baseline (mean)": y_pred_base_reg
}

reg_results = []
for name, preds in reg_models.items():
    reg_results.append(evaluate_regression(name, y_test_reg, preds))

reg_df = pd.DataFrame(reg_results)
print(reg_df.round(3))